# Clase 10: clustering no supervisado con K-Means

Piensa en todo lo que hiciste hasta la clase nueve: regresión, los k vecinos más cercanos (KNN) y la máquina de vectores de soporte (SVM). En los tres casos le dabas al algoritmo la respuesta correcta de cada ejemplo: el precio real, la clase real. ¿Y si no la tienes? ¿Y si nadie ha etiquetado tus datos todavía?

Hoy entras en un bloque distinto: el aprendizaje no supervisado. Le entregas los datos al algoritmo sin ninguna etiqueta y le pides que encuentre por sí mismo grupos de elementos parecidos. Vas a conocer K-Means, el algoritmo de clustering más usado en la práctica, y lo vas a aplicar sobre el conjunto de datos del Titanic para comprobar algo que sorprende la primera vez que lo ves: sin decirle nunca quién sobrevivió, el algoritmo agrupa a los pasajeros de una forma que se parece bastante a la supervivencia real.

¿Por qué te importa esto más allá de este ejercicio? Porque esta misma técnica sirve para segmentar clientes sin categorías previas, agrupar explotaciones agrarias por patrones de consumo o detectar perfiles de pacientes con comportamientos parecidos en un conjunto de datos de salud. Entender cómo funciona K-Means por dentro te permite usarlo con criterio, no como una caja mágica que escupe grupos.

## Aprendizaje no supervisado: agrupar sin etiquetas

Imagina que vuelcas sobre la mesa una caja de botones mezclados: de colores distintos, tamaños distintos, algunos con cuatro agujeros y otros con dos. Nadie te ha dicho de antemano cuántos grupos existen ni qué característica debes usar para separarlos. Aun así, tú, de forma natural, empiezas a acercar los botones parecidos entre sí: aquí los rojos, allí los grandes, más allá los de dos agujeros. No conocías las categorías antes de empezar; las descubriste mirando los propios botones.

Eso es exactamente lo que hace un algoritmo de aprendizaje no supervisado. No recibe ninguna etiqueta que le diga "este botón es del grupo A" y "este otro es del grupo B". Solo recibe las características de cada elemento y tiene que decidir, basándose en el parecido entre esas características, qué elementos van juntos.

El clustering es la familia de técnicas que resuelve este problema, y existen dos formas principales. ¿Tú decides de antemano cuántos grupos quieres, o dejas que el propio algoritmo lo averigüe?

- Clustering plano: tú decides de antemano cuántos grupos quieres, por ejemplo dos o tres, y el algoritmo reparte los datos en ese número exacto de grupos
- Clustering jerárquico: el propio algoritmo decide cuántos grupos existen y cómo se relacionan entre ellos, sin que tú fijes ese número

K-Means pertenece al primer tipo: tú eliges cuántos grupos, llamados clústeres, quieres encontrar, y el algoritmo busca los mejores centros posibles para esa cantidad. La letra k del nombre representa precisamente ese número de clústeres que tú decides antes de empezar.

La intuición central de K-Means es sencilla: coloca k puntos centrales, llamados centroides, e intenta que cada dato quede lo más cerca posible de su centroide asignado. El algoritmo mueve esos centroides paso a paso hasta que encajan bien con la forma real de los datos. Más adelante en esta misma clase ves el mecanismo exacto, paso a paso, cuando lo programas desde cero.

## Preparar datos con columnas de texto: pasar de categorías a números

K-Means, como casi todos los algoritmos de aprendizaje automático, solo entiende números. Calcula distancias entre puntos, y una distancia solo tiene sentido entre valores numéricos. El problema es que muchos conjuntos de datos reales incluyen columnas de texto: el sexo de una persona, el puerto donde embarcó o el color de un producto. ¿Cómo le das eso a un algoritmo que solo sabe restar y elevar al cuadrado?

Vas a trabajar con el conjunto de datos del Titanic, que recoge información de los pasajeros del barco: clase del billete, sexo, edad, tarifa pagada y si sobrevivieron o no, entre otras columnas. Usas una copia pública estable del conjunto de datos, en vez de la fuente original que ya no está disponible, y la cargas directamente desde una URL con pandas. Esta celda necesita conexión a internet, porque descarga el archivo en el momento en que la ejecutas.

In [ ]:
import pandas as pd

# Esta celda necesita conexión a internet: descarga el CSV en el momento
# de ejecutarla. Si la descarga tarda unos segundos, es normal, solo espera
# a que termine antes de seguir con la siguiente celda.
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(df.head())
print(df.dtypes)

Al revisar `df.dtypes` ves que columnas como `Pclass`, `Age`, `Fare` o `Survived` ya son numéricas, pero otras, como `Sex` o `Embarked`, son de tipo texto (`object`). Esas columnas de texto son las que necesitas convertir antes de aplicar K-Means.

La técnica más directa se llama codificación por etiquetas (*label encoding*, del inglés): a cada valor distinto de una columna de texto le asignas un número entero único. Por ejemplo, "female" pasa a ser cero y "male" pasa a ser uno. Puedes construir tú mismo esta conversión con un diccionario, sin depender de ninguna librería adicional.

In [ ]:
def convertir_columnas_texto_a_numero(df):
    df = df.copy()
    columnas = df.columns.values

    for columna in columnas:
        texto_a_numero = {}

        def convertir_valor(valor):
            return texto_a_numero[valor]

        # Solo tocamos columnas que no son ya numéricas
        if df[columna].dtype != "int64" and df[columna].dtype != "float64":
            valores_columna = df[columna].fillna(0).tolist()
            valores_unicos = set(valores_columna)

            contador = 0
            for valor_unico in valores_unicos:
                if valor_unico not in texto_a_numero:
                    texto_a_numero[valor_unico] = contador
                    contador += 1

            df[columna] = list(map(convertir_valor, df[columna].fillna(0)))

    # Rellenamos con cero cualquier valor numérico que siga vacío, como
    # ocurre con la edad de varios pasajeros: K-Means no puede trabajar
    # con huecos. Rellenar con cero es una solución simple para este
    # ejercicio; en un proyecto real valorarías rellenar con la media o
    # la mediana de la columna en vez de con cero
    df = df.fillna(0)

    return df


# Nos quedamos con columnas relevantes y descartamos las que no aportan
# información útil para agrupar pasajeros (nombre, identificador de billete...)
df_modelo = df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)
df_modelo = convertir_columnas_texto_a_numero(df_modelo)

print(df_modelo.head())
print(df_modelo.dtypes)

Con este proceso, `Sex` pasa a contener cero y uno, y `Embarked` pasa a contener un número entero por cada puerto de embarque.

**Ejecuta `df_modelo.dtypes` después de la conversión y comprueba que todas las columnas son ya `int64` o `float64`.** Ese es el requisito mínimo para poder aplicar K-Means. ¿Ves algún valor de texto que se te haya escapado? Si aparece, vuelve a revisar qué columnas metiste en `df_modelo`.

## Aplicar K-Means con scikit-learn sobre el Titanic

Con los datos ya convertidos a números, puedes aplicar `KMeans`, la clase de agrupamiento que scikit-learn ofrece en `sklearn.cluster`. Aquí llega la pregunta interesante de la clase de hoy: si le pides al algoritmo que separe a los pasajeros en dos grupos sin decirle nunca quién sobrevivió, ¿esos dos grupos coincidirán con la supervivencia real?

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Separamos la columna objetivo: la usamos solo para comparar, nunca
# para entrenar. Dársela al algoritmo sería hacer trampa.
y = df_modelo["Survived"]
X = df_modelo.drop(["Survived"], axis=1)

# K-Means calcula distancias, así que escalamos las características para
# que ninguna domine el resultado solo por tener valores más grandes
# (la tarifa, por ejemplo, llega a superar los 500, mientras que el sexo
# solo vale 0 o 1)
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

# Creamos el modelo con dos clústeres: buscamos si el algoritmo separa
# a los pasajeros en dos grupos por su cuenta
modelo = KMeans(n_clusters=2, n_init=10, random_state=42)
modelo.fit(X_escalado)

etiquetas = modelo.labels_

# Comparamos las etiquetas encontradas por el algoritmo con la
# supervivencia real. Como K-Means numera sus grupos de forma arbitraria
# (el grupo 0 no significa "no sobrevivió" por definición), probamos
# las dos correspondencias posibles y nos quedamos con la mejor
aciertos_directos = sum(etiquetas == y) / len(y)
aciertos_invertidos = sum(etiquetas != y) / len(y)
precision = max(aciertos_directos, aciertos_invertidos)

print(f"Coincidencia entre los clústeres y la supervivencia real: {precision:.2%}")

Al ejecutar este código ves que la coincidencia ronda el 69%, un resultado nada trivial teniendo en cuenta que el algoritmo nunca vio la columna `Survived`. Esto confirma algo importante sobre el clustering: aunque es una técnica no supervisada, en muchos conjuntos de datos reales los grupos que encuentra por similitud de características coinciden, al menos en parte, con categorías que sí importan en el mundo real, como la supervivencia. Columnas como `Sex` o `Pclass` influyen mucho en ese parecido, porque separan de forma bastante clara a los pasajeros en subgrupos: no es casualidad que "mujer y de primera clase" y "hombre y de tercera clase" describan perfiles con destinos muy distintos en el hundimiento.

Este resultado también recuerda algo que ya viste con la SVM: en la práctica es habitual usar clustering como paso previo a un problema supervisado. Primero agrupas los datos sin etiquetas para explorar patrones, y después, si te interesa, usas esos grupos como si fueran etiquetas para entrenar un clasificador.

## Programar K-Means desde cero: el algoritmo de Lloyd

Ya viste qué hace K-Means desde fuera. Pero, ¿qué pasa por dentro cuando llamas a `.fit()`? El procedimiento que sigue `KMeans` de scikit-learn se llama algoritmo de Lloyd, y es mucho más sencillo de lo que su nombre sugiere. Consta de cuatro pasos que se repiten en bucle:

1. Inicialización: eliges k puntos del conjunto de datos como centroides de partida
2. Asignación: cada punto del conjunto de datos se asigna al centroide más cercano
3. Actualización: cada centroide se recalcula como el promedio de todos los puntos que tiene asignados
4. Repetición: vuelves a los pasos 2 y 3 hasta que los centroides dejan de moverse de forma significativa

Vas a programar este algoritmo desde cero, sin usar `KMeans` de scikit-learn, sobre un conjunto de datos pequeño generado con `make_blobs`. Así ves con total claridad cómo se mueven los centroides en cada vuelta, porque además vas a guardar esa evolución para dibujarla al final.

In [ ]:
import numpy as np


class KMeansDesdeCero:
    def __init__(self, k=2, tolerancia=0.001, max_iteraciones=300):
        self.k = k
        self.tolerancia = tolerancia
        self.max_iteraciones = max_iteraciones

    def ajustar(self, datos):
        # Paso 1: inicialización. Tomamos los primeros k puntos del
        # conjunto de datos como centroides de partida
        self.centroides = {}
        for i in range(self.k):
            self.centroides[i] = datos[i]

        # Guardamos una copia de los centroides en cada vuelta, incluida
        # la inicial, para poder dibujar después cómo se desplazan
        self.historial_centroides = [dict(self.centroides)]

        for iteracion in range(self.max_iteraciones):
            self.clasificaciones = {i: [] for i in range(self.k)}

            # Paso 2: asignación. Cada punto se asigna al centroide
            # más cercano, medido con la distancia euclídea
            for punto in datos:
                distancias = [
                    np.linalg.norm(punto - self.centroides[centroide])
                    for centroide in self.centroides
                ]
                clasificacion = distancias.index(min(distancias))
                self.clasificaciones[clasificacion].append(punto)

            centroides_anteriores = dict(self.centroides)

            # Paso 3: actualización. Cada centroide pasa a ser el
            # promedio de los puntos que tiene asignados
            for clasificacion in self.clasificaciones:
                self.centroides[clasificacion] = np.average(
                    self.clasificaciones[clasificacion], axis=0
                )

            self.historial_centroides.append(dict(self.centroides))

            # Paso 4: comprobamos si los centroides ya dejaron de moverse
            convergencia = True
            for centroide in self.centroides:
                centroide_original = centroides_anteriores[centroide]
                centroide_actual = self.centroides[centroide]
                cambio = np.sum(
                    (centroide_actual - centroide_original) / centroide_original * 100.0
                )
                if abs(cambio) > self.tolerancia:
                    convergencia = False

            if convergencia:
                print(f"El algoritmo convergió en la iteración {iteracion + 1}")
                break

        return self

    def predecir(self, punto):
        distancias = [
            np.linalg.norm(punto - self.centroides[centroide])
            for centroide in self.centroides
        ]
        return distancias.index(min(distancias))

Fíjate en tres detalles del código:

- Los centroides y las clasificaciones se guardan en diccionarios indexados por el número de clúster, así puedes recorrerlos y actualizarlos de forma ordenada en cada vuelta del bucle
- El criterio de convergencia compara el cambio porcentual entre el centroide anterior y el nuevo; si ese cambio es menor que la tolerancia para todos los centroides, el algoritmo se detiene aunque no haya llegado al máximo de iteraciones
- El método `predecir` reutiliza exactamente la misma lógica del paso de asignación: calcula la distancia del punto nuevo a cada centroide ya entrenado y devuelve el más cercano

Ahora genera un conjunto de datos pequeño con `make_blobs`, la función de scikit-learn que crea grupos de puntos alrededor de unos centros, pensada justo para probar algoritmos de clustering.

In [ ]:
from sklearn.datasets import make_blobs

# Un conjunto de datos pequeño con tres grupos algo solapados, pensado
# para que el algoritmo necesite varias vueltas antes de converger
datos, _ = make_blobs(n_samples=90, centers=3, cluster_std=1.3, random_state=0)

modelo_propio = KMeansDesdeCero(k=3)
modelo_propio.ajustar(datos)

print("Centroides encontrados:")
for indice, centroide in modelo_propio.centroides.items():
    print(f"  Clúster {indice}: {centroide}")

punto_nuevo = np.array([0, 0])
print(f"El punto {punto_nuevo} pertenece al clúster {modelo_propio.predecir(punto_nuevo)}")

Ahora viene la parte que hace tangible todo lo anterior: dibuja cómo se desplazó cada centroide, vuelta a vuelta, desde su posición inicial hasta la posición final. Cada línea de color conecta las posiciones sucesivas de un mismo centroide, y la marca en forma de X señala dónde terminó.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
plt.scatter(datos[:, 0], datos[:, 1], c="lightgray", s=30, label="Puntos del dataset")

colores = ["red", "blue", "green", "purple", "orange"]
for indice_cluster in range(modelo_propio.k):
    # Recorremos el historial y sacamos la posición de este centroide en
    # cada vuelta, para dibujar su trayectoria completa
    trayectoria = np.array(
        [paso[indice_cluster] for paso in modelo_propio.historial_centroides]
    )
    color = colores[indice_cluster % len(colores)]
    plt.plot(
        trayectoria[:, 0], trayectoria[:, 1],
        marker="o", color=color, linestyle="--",
        label=f"Centroide {indice_cluster}",
    )
    plt.scatter(
        trayectoria[-1, 0], trayectoria[-1, 1],
        marker="X", s=200, color=color, edgecolors="black", zorder=5,
    )

plt.title("Evolución de los centroides durante el algoritmo de Lloyd")
plt.legend()
plt.show()

Si ejecutas este código verás el algoritmo converger en pocas iteraciones, porque el conjunto de datos de ejemplo tiene tres grupos razonablemente separados entre sí. Los centroides finales quedan cerca del centro real de cada grupo de puntos, y el gráfico anterior te deja ver exactamente el camino que siguieron para llegar hasta ahí: exactamente el comportamiento que esperabas del algoritmo de Lloyd.

## Resumen: lo que te llevas de esta clase

- El aprendizaje no supervisado busca patrones en datos sin etiquetas: nadie le dice al algoritmo la respuesta correcta de antemano
- K-Means es un algoritmo de clustering plano: tú decides con antelación el número de clústeres, k, que quieres encontrar
- Antes de aplicar K-Means necesitas convertir las columnas de texto a números, por ejemplo con codificación por etiquetas, y rellenar los valores ausentes
- Aplicado al Titanic sin la columna de supervivencia, K-Means agrupa a los pasajeros de una forma que coincide, en buena parte, con quién sobrevivió realmente
- El algoritmo de Lloyd, que scikit-learn usa por dentro, se reduce a repetir dos pasos: asignar cada punto al centroide más cercano y recalcular los centroides como el promedio de sus puntos asignados
- Programar K-Means desde cero confirma que el mecanismo interno es sencillo, aunque scikit-learn lo ejecuta de forma mucho más optimizada

**Ahora te toca a ti.** Abre el cuaderno de práctica de esta clase: allí completas el preprocesado de otra columna categórica del Titanic y programas tú mismo los pasos de asignación y recálculo de centroides sobre un dataset distinto.

**Siglas usadas en esta clase:** KNN (*K-Nearest Neighbors*, k vecinos más cercanos), SVM (*Support Vector Machine*, máquina de vectores de soporte).